In [1]:
import math
import copy
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from collections.abc import Callable
from typing import Any, List, Tuple, Any, Dict
from itertools import count
from importnb import Notebook

with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserRequest import UserRequestEvents
    from LabPrefetchScheduler import PrefetchScheduler

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.config as config
import Common.datatypes as datatypes
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(utils)

TransitionTuple = datatypes.TransitionTuple

In [ ]:
class EnvWrapper(gym.Env):
    """
    Simple wrapper that delegates all calls to an inner env.
    Subclass this to create your own wrappers.
    """

    metadata = {"render.modes": []}

    def __init__(
        self, 
        cfg: Any,
        n: int,
        m: int,
        n_layers: int,
        lam: float,
        theta: float,
        users_env: UserRequestEvents | None = None,
        du_caches: list | None = None,
        mec_cache: CacheEngineEnv | None = None,
        latency_model: MultiDULatencyModel | None = None,
        prefetch_fn: Callable | None = None,
        reward_fn: Callable | None = None,
        max_steps: int = 10000,
        *,
        step_duration_s: float = 1.0
    ):
        super().__init__()

        self.cfg = cfg 
        self.step_count = 0
        self.step_duration_s = step_duration_s
        self.max_steps = max_steps

        self.n = n  # number of tiles per row/column
        self.m = m  # number of tiles per row/column
        self.n_layers = n_layers  # number of layers (base + enhancement)
        
        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -1.0

        self.theta = theta
        self.lam = lam

        self.users_env = users_env
        self.du_caches = du_caches or []
        self.mec_cache = mec_cache
        self.latency_model = latency_model

        self.prefetch_fn = prefetch_fn or (lambda cache, action: cache.drl_prefetching(action))
        self.reward_fn = reward_fn or (lambda info: info.get('reward_per_user', {}).get(info.get('current_user', -1), 0.0))

        # History holders
        self.users_reward: Dict[int, list] = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr: Dict[int, list] = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user: Dict[int, int] = {
            u: 0 for u in range(self.users_env.n_users)
        }
        
        self.scheduler = PrefetchScheduler(
            R_M_D=self.latency_model.R_M_D,
            R_C_M=self.latency_model.R_C_M,
            U=self.latency_model.max_U,
            step_duration_s=self.step_duration_s
        )
        
    # ─────────────────────────────────────────────────────────────────────────
    # Internal Helpers
    # ─────────────────────────────────────────────────────────────────────────
    def _make_ready_bitmaps(self, du_planned: list | None, mec_planned: Any) -> Tuple[list | None, Any]:
        """Convert planned cache states to ready bitmaps with latency modeling."""
        du_ready = None
        mec_ready = None

        if du_planned:
            du_ready = [
                self.scheduler.materialize_ready_bitmap(f"DU:{du_idx}", bm)
                for du_idx, bm in enumerate(du_planned)
            ]

        if mec_planned is not None:
            mec_ready = self.scheduler.materialize_ready_bitmap("MEC", mec_planned)

        return du_ready, mec_ready
    
    def _has_item_miss(self, req: Dict[str, Any]) -> bool:
        """Check if request has cache miss for any required item."""
        gop = req["gop"]
        
        for tile in req["tiles"]:
            du_hit = tile["events"]["alpha_p_u"]
            mec_hit = tile["events"]["alpha_M_u"]
            layer = tile["layer"]
            
            # Miss if: no cache hit AND (base layer at gop 0 OR enhancement layer)
            if not (du_hit or mec_hit):
                match (gop, layer):
                    case (0, 0) | (_, 1):
                        return True

        return False

    def _has_base_layer(self, req: Dict[str, Any]) -> bool:
        """Check if request has base layer cached."""
        return any(
            tile["layer"] == 0 and tile["events"]["alpha_M_u"]
            for tile in req["tiles"]
        )

    def _missing_items(self, req: Dict[str, Any]) -> list[int]:
        """Identify which items (base + tiles) are missing from cache."""
        missing = [0] * 4

        # Check enhancement tiles (layer 1)
        for i, tile_id in enumerate(req["viewport"]):
            if not self.mec_cache.check_tile_in_cache(req["video"], 1, tile_id, 0):
                missing[i] = 1

        return missing

    def _create_transition(self, state: np.ndarray, action_idx: int, done: bool) -> TransitionTuple:
        """Create frozen DRL transition snapshot with deep copy to avoid mutation."""
        return TransitionTuple(
            state=copy.deepcopy(state),
            action=action_idx,
            reward=0.0,
            next_state=None,
            done=done
        )

    def _process_prefetch_actions(
        self,
        req: Dict[str, Any],
        meta: Any,
        ctrl: Any,
        net_adapter: Any,
        du_plan: list | None,
        mec_plan: Any
    ) -> Tuple[TransitionTuple, list[TransitionTuple | None], list | None, Any]:

        video = req["video"]
        viewport = req["viewport"]

        # -------- High-level goal selection --------
        meta_state = net_adapter.build_observation(req, video)
        goal = meta.select_goal(meta_state)

        # -------- Low-level loop to achieve goal --------
        action = {
            "video": video,
            "tiles": [], 
            "base_req_init": True, 
            "action_idx": goal,
        }
        
        # Apply to MEC cache
        mec_plan = self.prefetch_fn(self.mec_cache, action)

        transitions: list[TransitionTuple | None] = [None] * 4
        missing = self._missing_items(req)

        for idx, is_missing in enumerate(missing):
            if not is_missing:
                continue
    
            state = net_adapter.build_observation(req, video, viewport[idx])
                        
            # === Low-Level Action Selection ===
            action_idx = ctrl.select_action(state, goal)
            env_action = action_idx + 1 if action_idx < 4 else 0

            action = {
                "video": video,
                "tiles": [viewport[idx]],
                "base_req_init": False,
                "action_idx": env_action,
            }

            # Apply to MEC cache
            mec_plan = self.prefetch_fn(self.mec_cache, action)

            # Create immutable transition snapshot
            transitions[idx] = self._create_transition(state, action_idx, done=False)
        
        meta_transition = TransitionTuple(
            state=meta_state, 
            action=goal, 
            reward=0.0, 
            next_state=None, 
            done=self.users_env.all_users_done()
        )
        
        return meta_transition, transitions, du_plan, mec_plan

    def _recompute_cache_hits(self, req, du_bitmaps, mec_bitmap):
        """Update request events with cache hit/miss based on ready bitmaps."""
        for tile in req["tiles"]:
            layer = tile["layer"]
            tile_id = tile["tile"]

            # Check MEC cache
            mec_hit = mec_bitmap is not None and mec_bitmap[req["video"], layer, tile_id, 0] == 1

            tile["events"]["alpha_M_u"] = int(mec_hit)

    def _compute_cache_hits(self, req):
        video = req["video"]
        viewport = req["viewport"]

        video_cache_index = self.mec_cache.policy.video_idx
        tile_cache_index = self.mec_cache.policy.tile_idx

        base_hit = 1 if video in video_cache_index else 0
        
        video_idx = self.mec_cache.get_video_cache_idx(video)

        if video_idx != -1:
            cached_tiles = tile_cache_index[video_idx]
            enh_hit = [1 if tile in cached_tiles else 0 for tile in viewport]
        else:
            enh_hit = [0, 0, 0, 0]
    
        return dict(
            base_layer_hits=12 * base_hit,
            enh_layer_hits=sum(enh_hit),
            base_layer_misses=12 * (1 - base_hit),
            enh_layer_misses=4 - sum(enh_hit)
        )

    # ─────────────────────────────────────────────────────────────────────────
    # Gym Environment API
    # ─────────────────────────────────────────────────────────────────────────
    def sample_action(self) -> Tuple[Dict[str, Any], int]:
        """Sample random action with center viewport."""
        tiles = np.zeros(self.n * self.m, dtype=int)
        c = self.n // 2
        
        if self.n % 2 == 1:
            center_idx = c * self.n + c
            tiles[center_idx] = 1
        else:
            # Mark 2x2 center tiles
            for x, y in [(c-1, c-1), (c-1, c), (c, c-1), (c, c)]:
                tiles[y * self.n + x] = 1

        return {
            'video': np.random.randint(0, self.users_env.n_videos),
            'gop': np.random.randint(0, self.users_env.n_gops),
            'tiles': tiles.tolist()
        }, c * self.n + c

    def step(
        self, 
        meta: Any, 
        ctrl: Any, 
        net_adapter: Any, 
        req: list[Dict[str, Any]]
    ) -> Tuple[dict, float, bool, dict]:
        """Execute single environment step with DRL prefetching decisions."""
        info = {}

        du_plan = self.du_caches if len(self.du_caches) > 0 else None
        mec_plan = self.mec_cache.get_cache_bitmap() if self.mec_cache else None

        meta_transition, transitions, du_plan, mec_plan = self._process_prefetch_actions(
            req, meta, ctrl, net_adapter, du_plan, mec_plan
        )

        nxt_req = self.users_env.get_next_request(du_plan, mec_plan)

        video = nxt_req["video"]
        viewport = nxt_req["viewport"]
        
        net_adapter.features.update_ch_history(video, viewport)

        reward_val = net_adapter.features.compute_reward()

        meta_transition.next_state = net_adapter.build_observation(
            nxt_req, nxt_req["video"]
        )

        meta_transition.reward = reward_val 
        meta.remember( 
            meta_transition.state,
            meta_transition.action, 
            meta_transition.reward, 
            meta_transition.next_state, 
            done=self.users_env.all_users_done() 
        )

        for idx, trans in enumerate(transitions):
            if trans is None or nxt_req is None:
                continue

            # Build next state
            nxt_state = net_adapter.build_observation(
                nxt_req, nxt_req["video"], nxt_req["viewport"][idx]
            )

            trans.next_state = nxt_state
            transitions[idx].reward = reward_val

            ctrl.remember(
                transitions[idx].state,
                transitions[idx].action,
                transitions[idx].reward,
                transitions[idx].next_state,
                done=self.users_env.all_users_done()
            )
            ctrl.train_step()        
            meta.train_step()
    
        # -------------------------------------------------------
        # 2. Remaining users requests
        # -------------------------------------------------------
        info["user_request"] = req

        # -------------------------------------------------------
        # 3. Cache stats (HIT / MISS)
        # -------------------------------------------------------
        info.update(self._compute_cache_hits(req))

        done = self.users_env.all_users_done()
        self.step_count += 1

        return {}, reward_val, done, info



    # ---------------------------------------------------------
    #  LATENCY + BANDWIDTH COST
    # ---------------------------------------------------------
    def compute_latency_and_bw(self, reqs: list[Dict[str, Any]]) -> Tuple[Dict[int, float], Dict[int, float]]:
        """Compute latency and bandwidth cost per user based on cache hits/misses."""
        latency_per_user: Dict[int, float] = {u: 0.0 for u in range(self.users_env.n_users)}
        bw_cost_per_user: Dict[int, float] = {u: 0.0 for u in range(self.users_env.n_users)}

        for req in reqs:
            u = req["u"]

            for tile in req["tiles"]:
                layer = tile["layer"]
                size = self.mec_cache.tile_size_bytes[layer]
                du_hit = tile["events"]["alpha_p_u"]
                mec_hit = tile["events"]["alpha_M_u"]

                # Classify event location
                per_byte_latency = (
                    self.latency_model.TD_U if du_hit == 1
                    else self.latency_model.R_M_D if mec_hit == 1
                    else self.latency_model.R_C_M
                )

                latency_per_user[u] += per_byte_latency * size
                bw_cost_per_user[u] += size

        return latency_per_user, bw_cost_per_user

    # ---------------------------------------------------------
    #  PSNR MODEL
    # ---------------------------------------------------------
    def compute_psnr(self, reqs: list[Dict[str, Any]]) -> Dict[int, float]:
        """Compute PSNR per user based on layer hits."""
        psnr_per_user: Dict[int, float] = {}
        
        for req in reqs:
            u = req["u"]
            base_sum = sum(
                30.0 for tile in req["tiles"]
                if tile["layer"] == 0 and tile["events"]["alpha_M_u"] == 1
            )
            enh_sum = sum(
                10.0 for tile in req["tiles"]
                if tile["layer"] == 1 and tile["events"]["alpha_M_u"] == 1
            )
            psnr_per_user[u] = (base_sum / 12.0) + (enh_sum / 4.0)

        return psnr_per_user

    # ---------------------------------------------------------
    # REWARD FUNCTION
    # ---------------------------------------------------------
    def compute_reward(self, psnr_per_user: Dict[int, float]) -> Dict[int, float]:
        """Map PSNR to reward."""
        return psnr_per_user

    # ---------------------------------------------------------
    #  HIT / MISS STATS
    # ---------------------------------------------------------
    def compute_cache_stats(self, reqs: list[Dict[str, Any]]) -> Dict[str, int]:
        """Compute cache hit/miss statistics by layer."""
        stats = {"base_layer_hits": 0, "enh_layer_hits": 0, "base_layer_misses": 0, "enh_layer_misses": 0}
        
        for req in reqs:
            for tile in req["tiles"]:
                layer = tile["layer"]
                hit = int(tile["events"]["alpha_p_u"] or tile["events"]["alpha_M_u"])
                
                match layer:
                    case 0:
                        stats["base_layer_hits"] += hit
                        stats["base_layer_misses"] += (1 - hit)
                    case _:
                        stats["enh_layer_hits"] += hit
                        stats["enh_layer_misses"] += (1 - hit)

        return stats

    # ---------------------------------------------------------
    # RESET
    # ---------------------------------------------------------
    def reset(self, **kwargs) -> Tuple[None, Dict[str, Any]]:
        """Reset environment to initial state."""
        self.step_count = 0
        self.nxt_req = None
        
        self.users_reward = {u: [] for u in range(self.users_env.n_users)}
        self.users_psnr = {u: [] for u in range(self.users_env.n_users)}
        self.total_gop_requests_per_user = {u: 0 for u in range(self.users_env.n_users)}

        _, info_users = self.users_env.reset(**kwargs)
        info_cache_mec = self.mec_cache.reset(**kwargs)[1] if self.mec_cache else {}

        req = self.users_env.get_next_request(None, None)

        info_cache = {
            **info_cache_mec,
            **info_users,
            "user_request": req
        }

        self.scheduler.now_s = 0.0
        self.scheduler.availability = {}

        return None, info_cache

    def handle_base_layer_request(self, env, req, net_adapter, agent, cfg):
        """Handle base layer cache decision."""
        user = req["u"]
        video = req["video"]
        viewport = req["viewport"]

        state = net_adapter.build_observation(video)
        action_idx, _ = agent.select_action(state)

        should_cache = (action_idx == 1)

        if should_cache:
            cached_videos = env.mec_cache.policy.video_idx
            freqs = [
                net_adapter.features.video_freq_long.get(v, 0) if v != -1 else -1
                for v in cached_videos
            ]
            mec_action = np.argmin(freqs) + 1
        else:
            mec_action = 0

        action = {
            "user": user,
            "video": video,
            "tiles": viewport,
            "base_req_init": should_cache,
            "action_idx": mec_action
        }

        transition = datatypes.UserTransition(
            state=state,
            action=action_idx,
            reward=0.0,
            next_state=None
        )

        return action, transition

    def handle_existing_base_layer(self, env, req, net_adapter, cfg, action_idx=0):
        """Handle tile ranking when base layer exists."""
        user = req["u"]
        video = req["video"]
        viewport = list(req["viewport"])

        bitmap = env.mec_cache.get_cache_bitmap()
        cached_tiles = np.argwhere(bitmap[video, 1, :, 0] == 1).flatten().tolist()

        ranked = net_adapter.rank_viewport_tiles(
            video,
            viewport + cached_tiles,
            net_adapter.features,
            alpha=0.7
        )

        return {
            "user": user,
            "video": video,
            "tiles": ranked[:cfg.viewport],
            "base_req_init": False,
            "action_idx": action_idx
        }


In [3]:
def getTiles(step: int, user: int, users_viewport_tiles: Dict[int, list], n: int) -> np.ndarray:
    """Convert viewport tiles for user at step to binary mask."""
    mask = np.zeros(n * n, dtype=int)
    
    for tx, ty in users_viewport_tiles[user][step]:
        if 0 <= tx < n and 0 <= ty < n:
            mask[ty * n + tx] = 1
    
    return mask


In [4]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserRequestEvents(
        n_nodes=n_nodes,
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    # du_caches = [
    #     CacheEngineEnv(
    #         n_tiles=n*n,
    #         n_videos=n_videos,
    #         cache_capacity=max_capacity
    #     ) for _ in range(n_nodes)  # Number of DUs = n_nodes
    # ]
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward, done, info = env.step(actions)

        total_reward += float(reward)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break

TypeError: UserRequestEvents.__init__() got an unexpected keyword argument 'alpha'

In [ ]:
if __name__ == "__main__":
    steps = range(1, len(results) + 1)
    cache_hits_series = [r["cache_hits"] for r in results]
    cache_misses_series = [r["cache_misses"] for r in results]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)

    # Rewards with moving average
    axes[0].plot(steps, [r["total_reward"] for r in results], label="Step reward", alpha=0.7, color="blue")
    w = max(1, min(20, len(results) // 10))
    if w > 1:
        ma = [sum([r["total_reward"] for r in results][i - w:i]) / w for i in range(w, len(results) + 1)]
        axes[0].plot(range(w, len(results) + 1), ma, label=f"Moving avg (w={w})", color="orange")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Total reward")
    axes[0].set_title("Training Rewards")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Cache hits
    axes[1].plot(steps, cache_hits_series, label="Cache hits", color="green", alpha=0.8)
    axes[1].set_title("Cache Hits per Step")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Hits")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # Cache misses
    axes[2].plot(steps, cache_misses_series, label="Cache misses", color="red", alpha=0.8)
    axes[2].set_title("Cache Misses per Step")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Misses")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserTileRequestEvents(
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward, done, info = env.step(actions)

        total_reward += float(reward)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break